In [4]:
import matplotlib.pyplot as plt

In [4]:
import os
import yfinance as yf
import time
import datetime
import smtplib
from email.message import EmailMessage

def send_trade_notification(signal,price,pnl):
    msg = EmailMessage()
    msg.set_content(f'Update From The Bot:\n Action:{signal}\n price:{price}\n PnL{pnl}')
    msg['Subject'] = f'Bot Alert {signal}'
    msg['From'] = 'smtp.quant@gmail.com'
    msg['To'] = 'smtp.quant@gmail.com'
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
        smtp.login('smtp.quant@gmail.com','ozms ifvz tuql lzmh')
        smtp.send_message(msg)

last_buy_price = 0
last_signal = 'None'

if os.path.exists('trade_log.csv'):
    try:
        with open('trade_log.csv','r') as f:
            lines = f.readlines()
            if len(lines) > 1:
                last_line = lines[-1].split(',')
                last_signal = last_line[2]
                if last_signal == 'BUY':
                    last_buy_price = float(last_line[1])
        print(f'The Bot was recovered last signal was : {last_signal}')
    except:
        print('The bot was starting fresh')
print('The Bot was Online')

while True:
    try:
        data = yf.download('RELIANCE.NS',period = '1d',interval = '1m',progress = False)
        data['SMA_45'] = data['Close']['RELIANCE.NS'].rolling(window=45).mean()
        data['SMA_190'] = data['Close']['RELIANCE.NS'].rolling(window=190).mean()
        latest = data.iloc[-1]
        sma_45_val = latest['SMA_45'].item()
        sma_190_val = latest['SMA_190'].item()
        current_price = latest['Close']['RELIANCE.NS'].item()

        if sma_45_val > sma_190_val :
            current_signal = 'BUY'
        else:
            current_signal = 'SELL'
        if current_signal != last_signal :
            pnl = 0
            if current_signal == 'SELL' and last_buy_price:
                pnl = current_price - last_buy_price
                print(f'Trade Was Closed: PnL {pnl}')
            if current_signal == 'BUY':
                last_buy_price = current_price
                print(f'Entry: Buying at {current_price}')
            with open('trade_log.csv','a') as f:
                f.write(f'{time.ctime()},{current_price},{current_signal},{pnl}\n')
                try:
                    send_trade_notification(current_signal, current_price, pnl)
                    print('Notification Sent')
                except:
                    print('Notifaction Failed')
            last_signal = current_signal
        else:
            print(f'{time.ctime()}|{current_price}|{current_signal}|{pnl}')
    except Exception as e:
        print(f' System Alert {e}')
        time.sleep(10)
        continue
    time.sleep(60)

The Bot was recovered last signal was : SELL
The Bot was Online
Thu May  7 10:18:02 2026|1439.199951171875|SELL|0
Thu May  7 10:19:02 2026|1439.0|SELL|0
Thu May  7 10:20:02 2026|1440.0999755859375|SELL|0


KeyboardInterrupt: 

In [ ]:
import os
import yfinance as yf
import time
import smtplib
from email.message import EmailMessage



def send_trade_notification(signal, price, pnl):
    msg = EmailMessage()
    msg.set_content(f'Update From The Bot:\n Action:{signal}\n price:{price}\n PnL:{pnl}')
    msg['Subject'] = f'Bot Alert {signal}'
    msg['From'] = 'smtp.quant@gmail.com'
    msg['To'] = 'smtp.quant@gmail.com'
    
    # FIX 1: Corrected smtplib spelling
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
        smtp.login('smtp.quant@gmail.com', 'bjxq rsbe ocmn fobm')
        smtp.send_message(msg)
Stop_loss_pct = -0.01
Target_profit_pct = 0.02

last_buy_price = 0
last_signal = 'None'

if os.path.exists('trade_log.csv'):
    try:
        with open('trade_log.csv','r') as f:
            lines = f.readlines()
            if len(lines) > 1:
                last_line = lines[-1].split(',')
                last_signal = last_line[2].strip()
                if last_signal == 'BUY':
                    last_buy_price = float(last_line[1])
        print(f'The Bot was recovered last signal was : {last_signal}')
    except Exception as e:
        print(f'Memory recovery failed: {e}. Starting fresh.')
print('The Bot is Online')

while True:
    try:
        data = yf.download('RELIANCE.NS', period='5d', interval='1m', progress=False)
        data['SMA_45'] = data['Close']['RELIANCE.NS'].rolling(window=45).mean()
        data['SMA_190'] = data['Close']['RELIANCE.NS'].rolling(window=190).mean()
        latest = data.iloc[-1]
        sma_45_val = latest['SMA_45'].item()
        sma_190_val = latest['SMA_190'].item()
        current_price = latest['Close']['RELIANCE.NS'].item()

        if last_signal == 'Buy' and last_buy_price > 0:
            current_pnl_pct = (current_price - last_buy_price)/last_buy_price
            trigger_signal = None

            if stop_loss_pct <= current_pnl_pct:
                trigger_signal = 'STOP LOSS'
            elif target_profit_pct >= current_pnl_pct:
                trigger_signal = 'TAKE PROFIT'
                
            if trigger_signal:
                pnl = current_price - last_buy_price
                print(f'\n {trigger_signal} HIT! Force Selling at {current_price:.2f} | PnL: {current_pnl_pct:.2%}')

                with open('trade_log.csv', 'a') as f:
                    f.write(f'{time.ctime()},{current_price},{trigger_signal},{pnl}\n')
                try:
                    send_trade_notification(trigger_signal, current_price, pnl)
                    print('Emergency Notification Sent')
                except Exception as e:
                    print(f' Notification Failed: {e}')

                last_signal = 'COOLDOWN'
                last_buy_price = 0
                time.sleep(60)
                continue
                
        if sma_45_val > sma_190_val:
                current_signal = 'BUY'
        else:
            current_signal = 'SELL'

        if last_signal == 'COOLDOWN':
            if current_signal == 'SELL':
                print('\n Cooldown finished. Market trend shifted down. System reset.')
                last_signal = 'SELL'
            else:
                pnl_display = 0
                print(f'{time.ctime()} | {current_price:.2f} | Status: COOLDOWN (Waiting for trend reset)')
        elif current_signal != last_signal:
            pnl = 0
            if current_signal == 'SELL' and last_buy_price > 0:
                pnl = current_price - last_buy_price
                print(f'\n Trade Closed (SMA Signal): PnL {pnl:.2f}')
                last_buy_price = 0
            if current_signal == 'BUY':
                last_buy_price = current_price
                print(f'\n Entry: Buying at {current_price:.2f}')
                
            with open('trade_log.csv','a') as f:
                f.write(f'{time.ctime()},{current_price},{current_signal},{pnl}\n')
                
            try:
                send_trade_notification(current_signal, current_price, pnl)
                print('Notification Sent')
            except Exception as e:
                print(f' Notification Failed: {e}')
                
            last_signal = current_signal
            
        else:
            # Heartbeat print
            pnl_display = current_price - last_buy_price if last_signal == 'BUY' else 0
            print(f'{time.ctime()} | Price: {current_price:.2f} | Signal: {current_signal} | Open PnL: {pnl_display:.2f}')
            
    except Exception as e:
        print(f'\n System Alert: {e}')
        time.sleep(10)
        continue

    time.sleep(60)

In [17]:
import os
import yfinance as yf
import time
import smtplib
from email.message import EmailMessage

# ==========================================
# DAY 24: RISK MANAGEMENT PARAMETERS
# ==========================================
STOP_LOSS_PCT = -0.01   # Cut losses if position drops 1%
TAKE_PROFIT_PCT = 0.02  # Secure profits if position rises 2%

# --- 1. EMAIL FUNCTION ---
def send_trade_notification(signal, price, pnl):
    msg = EmailMessage()
    msg.set_content(f'Update From The Bot:\nAction: {signal}\nPrice: {price:.2f}\nPnL: {pnl:.2f}')
    msg['Subject'] = f'Bot Alert: {signal}'
    msg['From'] = 'smtp.quant@gmail.com'
    msg['To'] = 'smtp.quant@gmail.com'
    
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
        # DO NOT POST THIS PASSWORD ONLINE AGAIN! 
        smtp.login('smtp.quant@gmail.com', 'bjxq rsbe ocmn fobm')
        smtp.send_message(msg)

# --- 2. INITIALIZE MEMORY ---
last_buy_price = 0
last_signal = 'None'

if os.path.exists('trade_log.csv'):
    try:
        with open('trade_log.csv','r') as f:
            lines = f.readlines()
            if len(lines) > 1:
                last_line = lines[-1].split(',')
                last_signal = last_line[2].strip()
                if last_signal == 'BUY':
                    last_buy_price = float(last_line[1])
        print(f'🤖 Memory recovered. Last signal was: {last_signal}')
    except Exception as e:
        print(f'⚠️ Memory recovery failed: {e}. Starting fresh.')
print('🚀 Day 24 Quantum Engine: Online (Risk Mgmt Active)')

# --- 3. MAIN EXECUTION LOOP ---
while True:
    try:
        # Pull 5 days of data to ensure the SMA_190 can calculate properly
        data = yf.download('RELIANCE.NS', period='5d', interval='1m', progress=False)
        data['SMA_45'] = data['Close']['RELIANCE.NS'].rolling(window=45).mean()
        data['SMA_190'] = data['Close']['RELIANCE.NS'].rolling(window=190).mean()
        
        latest = data.iloc[-1]
        sma_45_val = latest['SMA_45'].item()
        sma_190_val = latest['SMA_190'].item()
        current_price = latest['Close']['RELIANCE.NS'].item()

        # ==========================================
        # DAY 24: THE RISK TRIPWIRE
        # ==========================================
        if last_signal == 'BUY' and last_buy_price > 0:
            current_pnl_pct = (current_price - last_buy_price) / last_buy_price
            trigger_signal = None
            
            # Check if we hit our risk limits
            if current_pnl_pct <= STOP_LOSS_PCT:
                trigger_signal = 'STOP_LOSS'
            elif current_pnl_pct >= TAKE_PROFIT_PCT:
                trigger_signal = 'TAKE_PROFIT'
                
            # Execute emergency sell if tripped
            if trigger_signal:
                pnl = current_price - last_buy_price
                print(f'\n🚨 {trigger_signal} HIT! Force Selling at {current_price:.2f} | PnL: {current_pnl_pct:.2%}')
                
                with open('trade_log.csv', 'a') as f:
                    f.write(f'{time.ctime()},{current_price},{trigger_signal},{pnl}\n')
                
                try:
                    send_trade_notification(trigger_signal, current_price, pnl)
                    print('📧 Emergency Notification Sent')
                except Exception as e:
                    print(f'⚠️ Notification Failed: {e}')
                
                # Put the bot in a cooldown state so it doesn't immediately buy again
                last_signal = 'COOLDOWN'
                last_buy_price = 0
                time.sleep(60)
                continue # Skip the rest of the loop for this minute
        # ==========================================

        # Core Decision Logic
        if sma_45_val > sma_190_val:
            current_signal = 'BUY'
        else:
            current_signal = 'SELL'

        # --- 4. EXECUTION LOGIC ---
        # If we are in COOLDOWN, we wait until the market naturally crosses to SELL to reset
        if last_signal == 'COOLDOWN':
            if current_signal == 'SELL':
                print('\n🔄 Cooldown finished. Market trend shifted down. System reset.')
                last_signal = 'SELL'
            else:
                pnl_display = 0
                print(f'{time.ctime()} | {current_price:.2f} | Status: COOLDOWN (Waiting for trend reset)')
                
        # Normal SMA Crossover Logic
        elif current_signal != last_signal:
            pnl = 0
            if current_signal == 'SELL' and last_buy_price > 0:
                pnl = current_price - last_buy_price
                print(f'\n💰 Trade Closed (SMA Signal): PnL {pnl:.2f}')
                last_buy_price = 0 # Reset memory
                
            if current_signal == 'BUY':
                last_buy_price = current_price
                print(f'\n📥 Entry: Buying at {current_price:.2f}')
                
            with open('trade_log.csv','a') as f:
                f.write(f'{time.ctime()},{current_price},{current_signal},{pnl}\n')
                
            try:
                send_trade_notification(current_signal, current_price, pnl)
                print('📧 Notification Sent')
            except Exception as e:
                print(f'⚠️ Notification Failed: {e}')
                
            last_signal = current_signal
            
        else:
            # Heartbeat print
            pnl_display = current_price - last_buy_price if last_signal == 'BUY' else 0
            print(f'{time.ctime()} | Price: {current_price:.2f} | Signal: {current_signal} | Open PnL: {pnl_display:.2f}')
            
    except Exception as e:
        print(f'\n⚠️ System Alert: {e}')
        time.sleep(10)
        continue
        
    time.sleep(60)

🤖 Memory recovered. Last signal was: BUY
🚀 Day 24 Quantum Engine: Online (Risk Mgmt Active)
Thu May  7 12:20:07 2026 | Price: 1442.10 | Signal: BUY | Open PnL: -1.00
Thu May  7 12:21:07 2026 | Price: 1443.30 | Signal: BUY | Open PnL: 0.20


KeyboardInterrupt: 